# EDA – LIAR Dataset

Objectifs :
- Comprendre la structure et la distribution des labels (6 classes LIAR).
- Étudier la longueur des énoncés (tokens, caractères).
- Analyser les principales métadonnées (partis, speakers, sujets).
- Préparer un jeu de données propre dans `data/traitees/` pour les modèles.

Références :
- William Yang Wang, “Liar, Liar Pants on Fire”: A New Benchmark Dataset for Fake News Detection (ACL 2017).
- Description du LIAR Dataset (structure des colonnes, splits).

In [ ]:
import pandas as pd
from pathlib import Path
import plotly.express as px

# Répertoires)
DATA_DIR = Path("../data")
BRUTES_DIR = DATA_DIR / "brutes"
FUSIONNES_DIR = DATA_DIR / "fusionnes"
TRAITEES_DIR = DATA_DIR / "traitees"

TRAITEES_DIR.mkdir(parents=True, exist_ok=True)

FUSIONNES_DIR, TRAITEES_DIR

(WindowsPath('../data/fusionnes'), WindowsPath('../data/traitees'))

## Chargement du jeu de données

In [45]:
df = pd.read_csv(FUSIONNES_DIR / "liar_unifie.csv")
df.head()

,id,label,statement,subject,speaker,job_title,state_info,party,barely_true_counts,false_counts,half_true_counts,mostly_true_counts,pants_on_fire_counts,context,split
0,2635.json,false,Says the Annies List political group supports ...,abortion,dwayne-bohac,State representative,Texas,republican,0.0,1.0,0.0,0.0,0.0,a mailer,train
1,10540.json,half-true,When did the decline of coal start? It started...,"energy,history,job-accomplishments",scott-surovell,State delegate,Virginia,democrat,0.0,0.0,1.0,1.0,0.0,a floor speech.,train
2,324.json,mostly-true,"Hillary Clinton agrees with John McCain ""by vo...",foreign-policy,barack-obama,President,Illinois,democrat,70.0,71.0,160.0,163.0,9.0,Denver,train
3,1123.json,false,Health care reform legislation is likely to ma...,health-care,blog-posting,NaN,NaN,none,7.0,19.0,3.0,5.0,44.0,a news release,train
4,9028.json,half-true,The economic turnaround started at the end of ...,"economy,jobs",charlie-crist,NaN,Florida,democrat,15.0,9.0,20.0,19.0,2.0,an interview on CNN,train


In [46]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12791 entries, 0 to 12790
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    12791 non-null  str    
 1   label                 12791 non-null  str    
 2   statement             12791 non-null  str    
 3   subject               12789 non-null  str    
 4   speaker               12789 non-null  str    
 5   job_title             9223 non-null   str    
 6   state_info            10040 non-null  str    
 7   party                 12789 non-null  str    
 8   barely_true_counts    12789 non-null  float64
 9   false_counts          12789 non-null  float64
 10  half_true_counts      12789 non-null  float64
 11  mostly_true_counts    12789 non-null  float64
 12  pants_on_fire_counts  12789 non-null  float64
 13  context               12660 non-null  str    
 14  split                 12791 non-null  str    
dtypes: float64(5), str(10)
memory 

In [47]:
df.isna().sum()

id                         0
label                      0
statement                  0
subject                    2
speaker                    2
job_title               3568
state_info              2751
party                      2
barely_true_counts         2
false_counts               2
half_true_counts           2
mostly_true_counts         2
pants_on_fire_counts       2
context                  131
split                      0
dtype: int64

In [48]:
# Copie brute figée pour référence
df.to_parquet(TRAITEES_DIR / "liar_unifie_raw.parquet", index=False)

## Nettoyage

On applique un nettoyage léger :
- suppression des lignes sans `statement`,
- normalisation simple du texte,
- remplacement des valeurs manquantes dans certaines métadonnées par `"unknown"`.

In [49]:
# Drop des lignes sans texte
df = df.dropna(subset=["statement"])

# Normalisation simple du texte
df["statement"] = df["statement"].astype(str).str.strip()
df = df[df["statement"].str.len() > 0]

# Gestion des NaN sur quelques colonnes meta (si elles existent)
meta_cols = ["speaker", "party", "subject", "state_info", "context", "job_title"]
for col in meta_cols:
    if col in df.columns:
        df[col] = df[col].fillna("unknown")

df.isna().sum()

id                      0
label                   0
statement               0
subject                 0
speaker                 0
job_title               0
state_info              0
party                   0
barely_true_counts      2
false_counts            2
half_true_counts        2
mostly_true_counts      2
pants_on_fire_counts    2
context                 0
split                   0
dtype: int64

In [50]:
df.to_parquet(TRAITEES_DIR / "liar_base_clean.parquet", index=False)

## Distribution des labels LIAR

Le dataset LIAR repose sur 6 labels de véracité :
`pants-fire`, `false`, `barely-true`, `half-true`, `mostly-true`, `true`.

On examine leur distribution dans le jeu unifié.

In [51]:
label_counts = df["label"].value_counts()
label_props = df["label"].value_counts(normalize=True)

label_counts, label_props

(label
 half-true      2627
 false          2507
 mostly-true    2454
 barely-true    2103
 true           2053
 pants-fire     1047
 Name: count, dtype: int64,
 label
 half-true      0.205379
 false          0.195997
 mostly-true    0.191854
 barely-true    0.164412
 true           0.160503
 pants-fire     0.081854
 Name: proportion, dtype: float64)

In [63]:
import plotly.express as px

df_plot = label_counts.reset_index(name='count').rename(columns={'index': 'label'})
df_plot['pct'] = (df_plot['count'] / df_plot['count'].sum() * 100).round(1).astype(str) + '%'

fig = px.bar(
    df_plot,
    x="label",
    y="count",
    text="pct",
    labels={"label": "label", "count": "count"},
    title="Distribution des labels LIAR (6 classes)"
)
fig.update_traces(textposition='outside')
fig.show()


## Mapping éventuel vers un label binaire

Pour certains modèles et pour faciliter la généralisation vers BuzzFeed (fake/real),
on peut définir un label binaire :

- 0 = plutôt faux : `pants-fire`, `false`, `barely-true`
- 1 = plutôt vrai : `half-true`, `mostly-true`, `true`.

On crée cette colonne `label_binary` sans encore l'utiliser dans les modèles.

In [53]:
mapping_binary = {
    "pants-fire": 0,
    "false": 0,
    "barely-true": 0,
    "half-true": 1,
    "mostly-true": 1,
    "true": 1,
}

df["label_binary"] = df["label"].map(mapping_binary)
df["label_binary"].value_counts(normalize=True)

label_binary
1    0.557736
0    0.442264
Name: proportion, dtype: float64

## Longueur des énoncés (tokens et caractères)

LIAR contient des déclarations politiques relativement courtes (ordre de 20 mots en moyenne).

On calcule :
- `n_chars` : longueur en caractères,
- `n_tokens` : nombre de tokens (split naïf sur les espaces).

In [54]:
df["n_chars"] = df["statement"].str.len()
df["n_tokens"] = df["statement"].str.split().str.len()

df[["n_chars", "n_tokens"]].describe()

,n_chars,n_tokens
count,12791.000000,12791.000000
mean,107.164022,18.040575
std,63.576726,10.129158
min,11.000000,2.000000
25%,73.000000,12.000000
50%,99.000000,17.000000
75%,133.000000,22.000000
max,3204.000000,467.000000


In [55]:
fig = px.histogram(
    df,
    x="n_tokens",
    nbins=50,
    title="Distribution de la longueur des énoncés (en tokens)"
)
fig.show()

In [56]:
fig = px.box(
    df,
    x="label",
    y="n_tokens",
    title="Longueur des énoncés (tokens) par label"
)
fig.show()

## Analyse des métadonnées (parti, speaker, sujet)

Le LIAR Dataset inclut des métadonnées riches : parti politique, speaker, sujet, état, contexte, etc.

On explore les distributions principales pour détecter d'éventuels déséquilibres et biais potentiels.

In [57]:
if "party" in df.columns:
    top_parties = df["party"].value_counts().head(10).reset_index(name='count').rename(columns={'index': 'party'})
    top_parties['pct'] = (top_parties['count'] / len(df) * 100).round(1).astype(str) + '%'
    fig = px.bar(
        top_parties,
        x="party",
        y="count",
        text="pct",
        labels={"party": "party", "count": "count"},
        title="Partis politiques les plus fréquents"
    )
    fig.update_traces(textposition='outside')
    fig.show()

    # Crosstab label x parti (proportions)
    crosstab_party = pd.crosstab(df["party"], df["label"], normalize="index")
    fig = px.imshow(
        crosstab_party,
        labels={"x": "label", "y": "party", "color": "proportion"},
        title="Répartition des labels par parti (proportions)"
    )
    fig.show()


In [58]:
if "party" in df.columns:
    top_5_parties = df["party"].value_counts().nlargest(5).index
    df_top_party = df[df["party"].isin(top_5_parties)]
    import plotly.express as px
    fig = px.box(df_top_party, x="party", y="n_tokens", title="Longueur des énoncés (tokens) par Parti (Top 5)")
    fig.show()

In [59]:
if "speaker" in df.columns:
    top_speakers = df["speaker"].value_counts().head(20).reset_index(name='count').rename(columns={'index': 'speaker'})
    top_speakers['pct'] = (top_speakers['count'] / len(df) * 100).round(1).astype(str) + '%'
    fig = px.bar(
        top_speakers,
        x="speaker",
        y="count",
        text="pct",
        labels={"speaker": "speaker", "count": "count"},
        title="Speakers les plus fréquents (Top 20)"
    )
    fig.update_traces(textposition='outside')
    fig.show()


In [60]:
if "subject" in df.columns:
    top_subjects = df["subject"].value_counts().head(15).reset_index(name='count').rename(columns={'index': 'subject'})
    top_subjects['pct'] = (top_subjects['count'] / len(df) * 100).round(1).astype(str) + '%'
    fig = px.bar(
        top_subjects,
        x="subject",
        y="count",
        text="pct",
        labels={"subject": "subject", "count": "count"},
        title="Sujets les plus fréquents (Top 15)"
    )
    fig.update_traces(textposition='outside')
    fig.show()


## Préparation des fichiers traités pour les modèles

On sauvegarde une version enrichie du LIAR unifié (`label_binary`, `n_tokens`, `n_chars`) et, si besoin, on définit des splits train/val/test dans `data/traitees/`.

Remarque : le LIAR original fournit des splits canoniques (train/val/test ≈ 10 269 / 1 284 / 1 283).
Puisque la colonne split a été enregistrée lors de la fusion, on retrouve ces splits de façon canonique.


In [61]:
df.to_parquet(TRAITEES_DIR / "liar_eda_features.parquet", index=False)

In [62]:
train = df[df["split"] == "train"].copy()
val   = df[df["split"] == "valid"].copy()
test  = df[df["split"] == "test"].copy()

for subset, name in [(train, "train"), (val, "val"), (test, "test")]:
    subset.to_parquet(TRAITEES_DIR / f"liar_{name}.parquet", index=False)

print("=== Distribution label_binary par Split ===")
print("\n--- Train ---")
print(train["label_binary"].value_counts(normalize=True))
print("\n--- Val ---")
print(val["label_binary"].value_counts(normalize=True))
print("\n--- Test ---")
print(test["label_binary"].value_counts(normalize=True))


=== Distribution label_binary par Split ===

--- Train ---
label_binary
1    0.561719
0    0.438281
Name: proportion, dtype: float64

--- Val ---
label_binary
1    0.520249
0    0.479751
Name: proportion, dtype: float64

--- Test ---
label_binary
1    0.563536
0    0.436464
Name: proportion, dtype: float64


## Synthèse EDA

- Le dataset LIAR unifié contient `12 791` lignes après nettoyage minimal (texte non vide).  
- La distribution des 6 labels est globalement relativement équilibrée, bien que `pants-fire` soit moins fréquent.
- La longueur moyenne des énoncés est d’environ 18 tokens, ce qui confirme qu’il s’agit de déclarations courtes plutôt que d’articles complets.  
- On observe des déséquilibres par parti et speaker (certains acteurs sont sur‑représentés), ce qui suggère un risque de biais de modèle.
- Ces observations guideront : 
  - le choix du framing (6 classes vs binaire),
  - la conception des modèles de base (TF‑IDF + linéaires),
  - l’analyse de biais dans `Interpretabilite_Biais.ipynb`.